## GPTS TimeSformer on UWHVF dataset

0. Import necessary packages

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# %cd /content/drive/MyDrive/Glaucoma Test
# !ls

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Dataloader import dataloaders
from sklearn.metrics import confusion_matrix, accuracy_score
import gc

Length of prog_labellist, total number of subjects with 3+ frames: 2991
Progressing number of subjects (1): 1769
Data preprocessing complete!
Loaded archetype_weights with shape: (2991, 18)
Total samples: 2991
Train size   : 2392
Test size    : 599


1. Preprocess data with data_preprocessing.py

In [2]:
%run data_preprocessing.py
!python compute_aa_weights-1.py

Length of prog_labellist, total number of subjects with 3+ frames: 2991
Progressing number of subjects (1): 1769
Data preprocessing complete!
Loaded archetype_weights with shape: (2991, 18)
Length of prog_labellist, total number of subjects with 3+ frames: 2991
Progressing number of subjects (1): 1769
Data preprocessing complete!
Loaded archetype_weights with shape: (2991, 18)
Alpha shape (n_samples, n_archetypes): (2991, 18)
Saved archetype weights to aa_weights.npy


In [3]:
from data_preprocessing import archetype_weights
NUM_ARCHETYPES = 18      # must match compute_aa_weights.py
print("archetype_weights is None:", archetype_weights is None)
if archetype_weights is not None:
    print("archetype_weights.shape:", archetype_weights.shape)
print(T)

archetype_weights is None: False
archetype_weights.shape: (2991, 18)


2. Set random seed for reproducability

In [4]:
torch.manual_seed(42)
np.random.seed(42)

2. Define Dataset class and DataLoaders with Dataloader.py

In [5]:
%run Dataloader.py

Total samples: 2991
Train size   : 2392
Test size    : 599


3. Define Model

In [6]:
#NUM_ARCHETYPES = 18      # must match compute_aa_weights.py
#!pip -q install timesformer-pytorch
%run Model.py
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/home/msi/anaconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


4. Make a folder to save model at the end of each epoch

In [7]:
!mkdir saved_models_GPTS_noAA
!mkdir saved_models_GPTS_LateFusionAA
!mkdir saved_models_GPTS_EarlyInjectionAA

In [ ]:
gc.collect()
torch.cuda.empty_cache()

5. Define optimizer, create instance of model, and enter training/validation loop with training_func.py

In [ ]:
%run training_func_noAA.py

Using device: cpu
Train dataset size: 2392
Train class counts (0=stable, 1=progressor): 977 1415

BASELINE (No AA) | Fold 1/5


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [11]:
#%run training_func_latefusionAA.py

Using device: cpu
Train dataset size: 2392
Train class counts (0=stable, 1=progressor): 977 1415

Fold 1/5
Fold 1 | Train size: 1913 | Val size: 479
Fold 1 | Epoch 1/100 | train_loss: 0.6812, train_acc: 0.5886 | val_loss: 0.6428, val_acc: 0.6180
Fold 1 | Epoch 2/100 | train_loss: 0.6567, train_acc: 0.6179 | val_loss: 0.7549, val_acc: 0.6409
Fold 1 | Epoch 3/100 | train_loss: 0.6472, train_acc: 0.6414 | val_loss: 0.6518, val_acc: 0.6326
Fold 1 | Epoch 4/100 | train_loss: 0.6522, train_acc: 0.6289 | val_loss: 0.6128, val_acc: 0.6743
Fold 1 | Epoch 5/100 | train_loss: 0.6430, train_acc: 0.6461 | val_loss: 0.6252, val_acc: 0.6576
Fold 1 | Epoch 6/100 | train_loss: 0.6475, train_acc: 0.6456 | val_loss: 0.6350, val_acc: 0.6806
Fold 1 | Epoch 7/100 | train_loss: 0.6433, train_acc: 0.6430 | val_loss: 0.6278, val_acc: 0.6597
Fold 1 | Epoch 8/100 | train_loss: 0.6413, train_acc: 0.6566 | val_loss: 0.6226, val_acc: 0.6639
Fold 1 | Epoch 9/100 | train_loss: 0.6351, train_acc: 0.6587 | val_loss: 0.

KeyboardInterrupt: 

In [ ]:
%run training_func_earlyinjectionAA.py

Using device: cpu
Train dataset size: 2392
Train class counts (0=stable, 1=progressor): 977 1415

Fold 1/5
Fold 1 | Train size: 1913 | Val size: 479


In [ ]:
gc.collect()
torch.cuda.empty_cache()

6. Run model with test data

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd  # Added for data saving
import os            # Added for directory management

from Dataloader import test_dataset
from Model import GPTS_NoAA, GPTS_LateFusionAA, GPTS_EarlyInjectionAA

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 5

# Create a directory to store plots if it doesn't exist
os.makedirs('./evaluation_results', exist_ok=True)

def evaluate_on_test(model, checkpoint_path: str, title: str):
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Load model state
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()

    true_labels, pred_labels, prob_scores = [], [], []

    with torch.no_grad():
        for data, aa_w, label in test_loader:
            data  = data.to(device).float()
            aa_w  = aa_w.to(device).float()
            label = label.unsqueeze(1).to(device)

            if isinstance(model, GPTS_NoAA):
                logits = model(data)
            else:
                logits = model(data, aa_w)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).cpu().numpy().astype(int)

            prob_scores.extend(probs.cpu().numpy().flatten())
            pred_labels.extend(preds.flatten())
            true_labels.extend(label.cpu().numpy().flatten())

    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)
    prob_scores = np.array(prob_scores)

    # Calculate Metrics
    acc  = accuracy_score(true_labels, pred_labels)
    prec = precision_score(true_labels, pred_labels, zero_division=0)
    rec  = recall_score(true_labels, pred_labels, zero_division=0)
    f1   = f1_score(true_labels, pred_labels, zero_division=0)

    try:
        auc = roc_auc_score(true_labels, prob_scores)
    except ValueError:
        auc = float("nan")

    # Console Output
    print(f"\n===== {title} =====")
    print(f"Accuracy : {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-score: {f1:.4f} | AUC: {auc:.4f}")

    # --- SAVE AND SHOW PLOTS ---
    file_safe_title = title.replace(" ", "_").lower()

    # Confusion matrix
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm',
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f"{title} - Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f'./evaluation_results/{file_safe_title}_cm.png') # Save plot
    plt.show()

    # ROC curve
    if not np.isnan(auc):
        fpr, tpr, _ = roc_curve(true_labels, prob_scores)
        plt.figure(figsize=(4, 3))
        plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.title(f"{title} - ROC Curve")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f'./evaluation_results/{file_safe_title}_roc.png') # Save plot
        plt.show()

    # Return metrics as a dictionary for saving to CSV
    return {
        "Model": title,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1_Score": f1,
        "AUC": auc
    }

# ---- Run Evaluations and Collect Results ----
all_results = []

# Baseline
all_results.append(evaluate_on_test(
    GPTS_NoAA(), 
    "./saved_models_GPTS_noAA/best_model_for_test.pth", 
    "Baseline (No AA)"
))

# Late Fusion
# all_results.append(evaluate_on_test(
#     GPTS_LateFusionAA(num_archetypes=18), 
#     "./saved_models_GPTS_LateFusionAA/best_model_for_test.pth", 
#     "Late Fusion AA"
# ))

# Early Injection
all_results.append(evaluate_on_test(
    GPTS_EarlyInjectionAA(num_archetypes=18), 
    "./saved_models_GPTS_EarlyInjectionAA/best_model_for_test.pth", 
    "Early Injection AA"
))

# ---- SAVE TO CSV ----
results_df = pd.DataFrame(all_results)
results_df.to_csv("./evaluation_results/test_metrics_summary.csv", index=False)
print("\nMetrics saved to ./evaluation_results/test_metrics_summary.csv")

In [ ]:
gc.collect()
torch.cuda.empty_cache()